# Tutorial 2A: AE and VAE training
Train a small autoencoder (AE) and variational autoencoder (VAE) on MNIST, each with a two-dimensional latent space that we can plot directly. Both reconstruct digits; the AE uses a deterministic code, while the VAE samples $z=\mu+\sigma\epsilon$ and adds a KL penalty toward $\mathcal N(0,I)$.

Teaching references: [PyTorch VAE example](https://github.com/pytorch/examples/blob/main/vae/main.py) and [Keras VAE tutorial](https://keras.io/examples/generative/vae/). See [setup instructions](../Instruction.md).

In [ ]:
import copy
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
train_data = datasets.MNIST("../data", train=True, download=True, transform=transforms.ToTensor())
test_data = datasets.MNIST("../data", train=False, download=True, transform=transforms.ToTensor())
indices = torch.randperm(len(train_data))[:10000].tolist()
train_loader = DataLoader(Subset(train_data, indices), batch_size=128, shuffle=True,
                          generator=torch.Generator().manual_seed(42))
fixed_digits = torch.stack([test_data[i][0] for i in range(8)]).to(device)

class DigitAutoencoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU())
        self.mean = nn.Linear(256, latent_dim)
        self.logvar = nn.Linear(256, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 256), nn.ReLU(),
                                     nn.Linear(256, 784), nn.Sigmoid())

    def decode(self, z):
        return self.decoder(z).view(-1, 1, 28, 28)

    def forward(self, x, variational=False):
        h = self.encoder(x)
        mu, logvar = self.mean(h), self.logvar(h)
        # AE uses mu directly; only the VAE learns and uses the variance head.
        z = mu
        if variational:
            epsilon = torch.randn_like(mu)
            z = mu + torch.exp(0.5 * logvar) * epsilon
        return self.decode(z), mu, logvar

### Training: reconstruction versus reconstruction + KL
The AE minimizes pixel reconstruction loss. The VAE adds
$D_{KL}=\frac12\sum_j(\mu_j^2+\exp(\mathrm{logvar}_j)-1-\mathrm{logvar}_j)$.

We use binary cross entropy as a simple reconstruction objective on MNIST intensities in [0, 1]. Both losses are summed over pixels/latent dimensions, then averaged over images. At evaluation we decode the VAE's posterior mean, so the before/after comparison reflects learning rather than sampling noise.

In [ ]:
# Re-run this cell to start both models from the same initial weights.
torch.manual_seed(42)
train_loader.generator.manual_seed(42)
ae = DigitAutoencoder().to(device)
digit_vae = copy.deepcopy(ae)
models = {"AE": ae, "VAE": digit_vae}
optimizers = {name: torch.optim.Adam(model.parameters(), lr=1e-3)
              for name, model in models.items()}
epochs = 10
history = {name: {"reconstruction": [], "KL": []} for name in models}
snapshots = {}

@torch.no_grad()
def record_reconstructions(epoch):
    for name, model in models.items():
        model.eval()
        snapshots[name, epoch] = model(fixed_digits, variational=False)[0].cpu()

record_reconstructions(0)
for epoch in range(1, epochs + 1):
    totals = {name: [0.0, 0.0] for name in models}
    count = 0
    for model in models.values():
        model.train()
    for x, _ in train_loader:
        x = x.to(device)
        for name, model in models.items():
            reconstruction, mu, logvar = model(x, variational=(name == "VAE"))
            rec_loss = F.binary_cross_entropy(reconstruction, x, reduction="none").flatten(1).sum(1).mean()
            kl = -0.5 * (1 + logvar - mu.square() - logvar.exp()).sum(1).mean() if name == "VAE" else x.new_zeros(())
            loss = rec_loss + kl
            optimizers[name].zero_grad()
            loss.backward()
            optimizers[name].step()
            totals[name][0] += rec_loss.item() * len(x)
            totals[name][1] += kl.item() * len(x)
        count += len(x)
    for name in models:
        history[name]["reconstruction"].append(totals[name][0] / count)
        history[name]["KL"].append(totals[name][1] / count)
    record_reconstructions(epoch)
    print(f"Epoch {epoch}: AE reconstruction={totals['AE'][0]/count:.1f}; "
          f"VAE reconstruction={totals['VAE'][0]/count:.1f}, KL={totals['VAE'][1]/count:.1f}")

### Exercise 1: Compare learning and generation
1. Compare the same eight test digits before training, after one epoch, and after the final epoch. What improves for each model?
2. Locate the sampling expression in `forward`. How does the VAE's training path differ from the AE's deterministic code?
3. Compare reconstruction curves and the VAE's KL curve. What does each objective encourage? Must KL decrease every epoch as the model learns to retain image information?

In [ ]:
def show_rows(rows, labels, title):
    fig, axes = plt.subplots(len(rows), 8, figsize=(10, 1.25 * len(rows)), squeeze=False)
    for row_axes, images, label in zip(axes, rows, labels):
        for ax, image in zip(row_axes, images):
            ax.imshow(image.detach().cpu().squeeze(), cmap="gray", vmin=0, vmax=1)
            ax.set_xticks([])
            ax.set_yticks([])
        row_axes[0].set_ylabel(label, rotation=0, ha="right", va="center")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

rows, labels = [fixed_digits], ["Original"]
for epoch in sorted({0, 1, epochs}):
    for name in models:
        rows.append(snapshots[name, epoch])
        labels.append(f"{name}, epoch {epoch}")
show_rows(rows, labels, "The same held-out digits throughout training")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for name in models:
    axes[0].plot(range(1, epochs + 1), history[name]["reconstruction"], marker="o", label=name)
axes[0].set_title("Training reconstruction loss per image")
axes[0].legend()
axes[1].plot(range(1, epochs + 1), history["VAE"]["KL"], marker="o")
axes[1].set_title("VAE KL per image")
for ax in axes:
    ax.set_xlabel("Epoch")
plt.tight_layout()
plt.show()

### Exercise 1b: Map the latent space
The models now have two latent coordinates. We plot the codes of 2,000 held-out digits, coloured by their true labels, and decode a grid.

The upper plots show AE codes and VAE posterior means on the same axis limits. The dashed square marks the region decoded below. VAE posterior means need not themselves follow a standard Gaussian: the KL term acts on the full posterior distributions.

**Questions:**
1. Where do the encoded digits lie relative to the dashed square? Which regions are visited by the encoder?
2. In the decoded grids, where are recognizable digits produced? How does this relate to the encoded points above?
3. Why does the AE's reconstruction objective not require useful outputs throughout this Gaussian sampling region? How does the VAE's KL term change the objective?

**In-lab question:** Replace the network with a CNN (Convolutional Neural Network) and analyze the differences.

Use the observations to discuss sampling, not to assume the VAE must win. The AE may arrange useful codes outside this box, and the same coordinates can represent different digits in the two models. A two-dimensional bottleneck also limits reconstruction quality.


In [ ]:
from matplotlib.patches import Rectangle

# Encode the same held-out digits for both models, using posterior means.
latent_loader = DataLoader(Subset(test_data, range(2000)), batch_size=128, shuffle=False)
encoded = {name: [] for name in models}
digit_labels = []
with torch.no_grad():
    for model in models.values():
        model.eval()
    for digits, labels in latent_loader:
        digits = digits.to(device)
        digit_labels.append(labels)
        for name, model in models.items():
            codes = model.mean(model.encoder(digits))
            encoded[name].append(codes.cpu())
encoded = {name: torch.cat(batches) for name, batches in encoded.items()}
digit_labels = torch.cat(digit_labels)

grid_size = 15
bounds = {
    name: (-6, 0, -6, 0) if name.upper() == "AE" else (-3, 3, -3, 3)
    for name in models
}
grid_images = {}

with torch.no_grad():
    for name, model in models.items():
        xmin, xmax, ymin, ymax = bounds[name]
        xs = torch.linspace(xmin, xmax, grid_size, device=device)
        ys = torch.linspace(ymax, ymin, grid_size, device=device)
        y, x = torch.meshgrid(ys, xs, indexing="ij")
        grid_codes = torch.stack([x.flatten(), y.flatten()], dim=1)

        digits = model.decode(grid_codes).cpu().squeeze(1)
        grid_images[name] = (
            digits.reshape(grid_size, grid_size, 28, 28)
            .permute(0, 2, 1, 3)
            .reshape(grid_size * 28, grid_size * 28)
        )

# Shared scatter limits include all codes and both sampling regions.
all_codes = torch.cat(list(encoded.values()))
low = min(all_codes.min().item(), min(min(b) for b in bounds.values())) - 0.5
high = max(all_codes.max().item(), max(max(b) for b in bounds.values())) + 0.5

fig, axes = plt.subplots(2, 2, figsize=(12, 11), constrained_layout=True)
for col, name in enumerate(models):
    xmin, xmax, ymin, ymax = bounds[name]
    codes = encoded[name]
    points = axes[0, col].scatter(
        codes[:, 0], codes[:, 1], c=digit_labels,
        cmap="tab10", vmin=-0.5, vmax=9.5, s=5, alpha=0.7
    )
    axes[0, col].add_patch(Rectangle(
        (xmin, ymin), xmax - xmin, ymax - ymin,
        fill=False, edgecolor="black", linestyle="--", linewidth=1.5
    ))
    axes[0, col].set(
        xlim=(low, high), ylim=(low, high), aspect="equal",
        title=f"{name}: encoded test digits",
        xlabel="Latent coordinate 1", ylabel="Latent coordinate 2"
    )

    # Pad by half a grid step to align coordinates with tile centres.
    half_x = (xmax - xmin) / (2 * (grid_size - 1))
    half_y = (ymax - ymin) / (2 * (grid_size - 1))
    axes[1, col].imshow(
        grid_images[name], cmap="gray", vmin=0, vmax=1,
        extent=(xmin - half_x, xmax + half_x,
                ymin - half_y, ymax + half_y),
        origin="upper"
    )
    axes[1, col].set(
        title=f"{name}: decoded grid",
        xlabel="Latent coordinate 1", ylabel="Latent coordinate 2"
    )
    axes[1, col].set_xticks([xmin, (xmin + xmax) / 2, xmax])
    axes[1, col].set_yticks([ymin, (ymin + ymax) / 2, ymax])

fig.colorbar(
    points, ax=list(axes[0]), ticks=range(10),
    label="Digit label", shrink=0.7
)
plt.show()

# Tutorial 2B: The computational benefit of latent diffusion
A pretrained image VAE compresses a photograph before the diffusion model processes it. We compare tensor storage and repeated convolution work at the two spatial resolutions. This VAE is separate from the MNIST models in Part 1.

### Exercise 2B.1: How much smaller is the representation?
Encode the image below. Compare the image and latent shapes, scalar counts, and storage. Distinguish spatial compression from `scale`, which changes latent amplitudes only.

In [ ]:
import torch
from pathlib import Path
from torchvision.transforms.functional import to_tensor
from diffusers import AutoencoderKL
from diffusers.utils import load_image
from IPython.display import display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device.type == "cuda" else torch.float32
repo_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
vae = AutoencoderKL.from_pretrained(repo_id, subfolder="vae", torch_dtype=dtype,
                                  use_safetensors=True).to(device).eval()
image_path = Path("../data/cat.png")
image = load_image(str(image_path) if image_path.exists() else
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/cat.png")
image = image.convert("RGB").resize((512, 512))
pixels = (to_tensor(image).unsqueeze(0) * 2 - 1).to(device=device, dtype=dtype)
scale = vae.config.scaling_factor

In [ ]:
from PIL import Image
import numpy as np

@torch.no_grad()
def pil_to_latents(image):
    init_image = to_tensor(image).unsqueeze(0) * 2 - 1
    init_image = init_image.to(device=device, dtype=dtype)
    posterior = vae.encode(init_image).latent_dist
    return posterior.sample() * scale

@torch.no_grad()
def latents_to_pil(latents):
    image = vae.decode(latents / scale).sample
    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.float().cpu().permute(0, 2, 3, 1).numpy()
    return [Image.fromarray((item * 255).round().astype(np.uint8)) for item in image]


latents = pil_to_latents(image)
print("Pixels:", tuple(pixels.shape), "Latents:", tuple(latents.shape))
display(image, latents_to_pil(latents)[0])

In [ ]:
import matplotlib.pyplot as plt
pixel_sites = pixels.shape[-2] * pixels.shape[-1]
latent_sites = latents.shape[-2] * latents.shape[-1]
representations = {"Pixels": pixels, "Latents": latents}
storage_kib = [tensor.numel() * tensor.element_size() / 1024 for tensor in representations.values()]
for name, tensor in representations.items():
    print(f"{name}: {tuple(tensor.shape)}, {tensor.numel():,} scalars, "
          f"{tensor.numel() * tensor.element_size() / 1024:.1f} KiB")
print(f"Spatial sites reduced by {pixel_sites / latent_sites:.0f}x")
print(f"Tensor storage reduced by {storage_kib[0] / storage_kib[1]:.0f}x at the same dtype")
plt.figure(figsize=(5, 3))
plt.bar(list(representations), storage_kib)
plt.ylabel("Tensor storage (KiB)")
plt.title("Representation size, not total model memory")
plt.tight_layout()
plt.show()

### Exercise 2B.2: Computational savings and reconstruction quality

**Quality check:** Compare the original image with its reconstruction after VAE encoding and decoding. We use the posterior mean for a repeatable comparison and measure errors on RGB values in [0, 1]. Lower MSE/MAE and higher PSNR indicate closer pixel reconstruction.
Inspect the error map as well as the numbers. Which details change? Does the example retain useful visual quality despite the smaller representation? Try another image before generalizing. VAE compression is by definition lossy.


In [ ]:

# Measure image reconstruction quality separately from the block benchmark.
with torch.no_grad():
    quality_posterior = vae.encode(pixels).latent_dist
    quality_reconstruction = vae.decode(quality_posterior.mode()).sample
    reference_rgb = (pixels.float() / 2 + 0.5).clamp(0, 1)
    reconstructed_rgb = (quality_reconstruction.float() / 2 + 0.5).clamp(0, 1)
    absolute_error = (reference_rgb - reconstructed_rgb).abs()
    reconstruction_mse = absolute_error.square().mean().item()
    reconstruction_mae = absolute_error.mean().item()
    reconstruction_psnr = -10 * np.log10(reconstruction_mse) if reconstruction_mse > 0 else float("inf")
    tolerance = 5 / 255
    pixels_within_tolerance = (absolute_error <= tolerance).all(dim=1).float().mean().item() * 100

print(f"Reconstruction MSE: {reconstruction_mse:.6f} (lower is better)")
print(f"Reconstruction MAE: {reconstruction_mae:.6f} (lower is better)")
print(f"Reconstruction PSNR: {reconstruction_psnr:.2f} dB (higher is better)")
print(f"Pixels within +/-5/255 in all RGB channels: {pixels_within_tolerance:.1f}%")

fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
axes[0].imshow(reference_rgb[0].cpu().permute(1, 2, 0))
axes[0].set_title("Original")
axes[1].imshow(reconstructed_rgb[0].cpu().permute(1, 2, 0))
axes[1].set_title(f"VAE reconstruction\nPSNR={reconstruction_psnr:.2f} dB")
error_map = absolute_error[0].mean(dim=0).cpu()
heatmap = axes[2].imshow(error_map, cmap="magma", vmin=0)
axes[2].set_title("Absolute error, averaged over RGB")
fig.colorbar(heatmap, ax=axes[2], label="Error on [0, 1] pixel scale", shrink=0.8)
for ax in axes:
    ax.axis("off")
plt.show()


### Using latent diffusion to generate an image from text
We reuse the VAE with its pretrained denoiser and text encoder. The pipeline denoises a latent tensor, then decodes it into pixels.

**Exercise 2B.3:** Change the prompt. Identify the impact and role of the text encoder, U-Net, scheduler (DDPM vs DDIM), and VAE. Why does the diffusion process operate on latents?

**Bonus:** Take a generated image, and invert it to get it's latents, can you invert both a DDIM and DDPM image?

In [ ]:
from diffusers import StableDiffusionPipeline, DDIMScheduler
ldm = StableDiffusionPipeline.from_pretrained(repo_id, vae=vae, torch_dtype=dtype,
                                            use_safetensors=True).to(device)
ldm.scheduler = DDIMScheduler.from_config(ldm.scheduler.config)
ldm.enable_attention_slicing()
prompt = "A painting of a squirrel eating a burger"
images = ldm(prompt, num_inference_steps=30, eta=0.0, guidance_scale=7.5,
             generator=torch.Generator(device=device).manual_seed(42)).images
display(images[0])

# Tutorial 3: Learn to move noise into data
We will train a small flow-matching model to turn Gaussian noise into a two-moons distribution. Each point is a two-dimensional sample, so we can see the entire generation process.

### Exercise 3.1: What should the model learn?
Look at the two distributions. The task is to transform a cloud of random points into the two curved bands, without being given a target point during generation.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

flow_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(7)
moon_points, _ = make_moons(n_samples=5000, noise=0.05, random_state=7)
flow_data = (torch.tensor(moon_points, dtype=torch.float32) - torch.tensor([0.5, 0.25])) * 2
flow_data = flow_data.to(flow_device)
fixed_noise = torch.randn(1000, 2, device=flow_device)

def plot_cloud(ax, points, title):
    points = points.detach().cpu()
    ax.scatter(points[:, 0], points[:, 1], s=3, alpha=0.5)
    ax.set(xlim=(-4, 4), ylim=(-4, 4), aspect="equal", title=title)
    ax.set_xticks([])
    ax.set_yticks([])

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
plot_cloud(axes[0], fixed_noise, "Starting distribution: Gaussian noise")
plot_cloud(axes[1], flow_data[:1000], "Target distribution: two moons")
plt.tight_layout()
plt.show()

### Exercise 3.2: Construct a velocity target and train
Pair a noise point $x_0$ with a randomly selected data point $x_1$. At a sampled time $t$, construct

$$x_t=(1-t)x_0+t x_1,\qquad u=x_1-x_0.$$

The arrow $u$ describes how the paired point moves along its straight training path. The network sees only $(x_t,t)$ and predicts velocity; the loss is $\|v_\theta(x_t,t)-u\|^2$.

**Questions:** Identify the input and target in the training loop. Why can different pairs give different target velocities near the same position and time? What does minimizing MSE encourage the network to predict there?

The first plot shows a few training pairs. The loss curve then shows learning. These straight pairwise paths define training targets; the learned sampling trajectories need not be straight.

In [ ]:
# Visualize a few conditional training targets before fitting the network.
with torch.no_grad():
    pair_noise = fixed_noise[:12]
    pair_data = flow_data[torch.randperm(len(flow_data), device=flow_device)[:12]]
    pair_midpoint = 0.5 * pair_noise + 0.5 * pair_data
    pair_velocity = pair_data - pair_noise
fig, ax = plt.subplots(figsize=(5, 4))
for a, b in zip(pair_noise.cpu(), pair_data.cpu()):
    ax.plot([a[0], b[0]], [a[1], b[1]], color="gray", alpha=0.3)
ax.scatter(*pair_noise.cpu().T, label="Noise endpoints", marker="x")
ax.scatter(*pair_data.cpu().T, label="Data endpoints")
ax.quiver(*pair_midpoint.cpu().T, *pair_velocity.cpu().T,
          angles="xy", scale_units="xy", scale=2, color="black", width=0.005)
ax.set(xlim=(-4, 4), ylim=(-4, 4), aspect="equal",
       title="Training pairs: velocity arrows at t=0.5 (half length)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Fresh initialization and fresh random pairs on each training iteration.
torch.manual_seed(42)
velocity_net = nn.Sequential(nn.Linear(3, 128), nn.SiLU(), nn.Linear(128, 128),
                             nn.SiLU(), nn.Linear(128, 128), nn.SiLU(), nn.Linear(128, 2)).to(flow_device)
flow_optimizer = torch.optim.Adam(velocity_net.parameters(), lr=1e-3)
training_steps = 5000
flow_losses = []
velocity_net.train()
for iteration in range(training_steps):
    x_1 = flow_data[torch.randint(len(flow_data), (256,), device=flow_device)]
    x_0 = torch.randn_like(x_1)
    t = torch.rand(len(x_1), 1, device=flow_device)
    x_t = (1 - t) * x_0 + t * x_1
    target_velocity = x_1 - x_0
    predicted_velocity = velocity_net(torch.cat([x_t, t], dim=1))
    loss = F.mse_loss(predicted_velocity, target_velocity)
    flow_optimizer.zero_grad()
    loss.backward()
    flow_optimizer.step()
    flow_losses.append(loss.item())
    if (iteration + 1) % 500 == 0:
        print(f"Iteration {iteration + 1}: mean loss over last 100 = {sum(flow_losses[-100:])/100:.3f}")

plt.figure(figsize=(6, 2.5))
plt.plot(range(100, training_steps + 1, 100),
         [sum(flow_losses[i:i+100])/100 for i in range(0, training_steps, 100)])
plt.xlabel("Training iteration")
plt.ylabel("Velocity MSE (100-batch mean)")
plt.tight_layout()
plt.show()

### Exercise 3.3: Watch the learned flow generate samples
Start from the same Gaussian cloud and repeatedly update each point using the predicted velocity. The midpoint solver below evaluates velocity twice per step. It receives **no target data points**.

**Questions:** At what stage do the two bands emerge? Compare 5 and 50 integration steps using the same initial noise. What changes in shape and computation? Inspect individual trajectories: are they the straight lines used to construct training targets?

**Bonus:** Do the same but learn the same data with DDPM, compare sampling steps until convergence, what do you notice? 

In [ ]:
@torch.no_grad()
def sample_flow(initial_noise, steps=50):
    if steps < 1:
        raise ValueError("Use at least one integration step")
    velocity_net.eval()
    state = initial_noise.clone()
    trajectory = [state.cpu().clone()]
    dt = 1.0 / steps
    for step in range(steps):
        time = torch.full((len(state), 1), step * dt, device=flow_device)
        velocity = velocity_net(torch.cat([state, time], dim=1))
        midpoint = state + 0.5 * dt * velocity
        state = state + dt * velocity_net(torch.cat([midpoint, time + 0.5 * dt], dim=1))
        trajectory.append(state.cpu().clone())
    return torch.stack(trajectory)

integration_steps = 50  # Try 5; no retraining is needed.
flow_trajectory = sample_flow(fixed_noise, steps=integration_steps)
fig, axes = plt.subplots(1, 6, figsize=(14, 3))
for ax, fraction in zip(axes[:5], [0, 0.25, 0.5, 0.75, 1]):
    index = round(fraction * integration_steps)
    plot_cloud(ax, flow_trajectory[index], f"Learned flow, t={index/integration_steps:.2f}")
plot_cloud(axes[-1], flow_data[:1000], "Target (for comparison only)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(*flow_data[:1000].cpu().T, s=2, color="gray", alpha=0.15)
for i in range(20):
    ax.plot(flow_trajectory[:, i, 0], flow_trajectory[:, i, 1], alpha=0.8)
ax.scatter(*flow_trajectory[0, :20].T, marker="x", color="black", label="Start: noise")
ax.scatter(*flow_trajectory[-1, :20].T, s=15, color="black", label="End: generated")
ax.set(xlim=(-4, 4), ylim=(-4, 4), aspect="equal", title="20 learned sampling trajectories")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"{integration_steps} integration steps = {2 * integration_steps} network evaluations")

### How does flow matching differ from diffusion?
Both methods learn to turn noise into data through a sequence of updates. The useful distinction in our labs is **the training target and how the prediction updates the sample**.

**Diffusion in Lab 1: learn to predict added noise.** We choose a clean image $x$, a noise sample $\epsilon$, and a timestep $t$. The scheduler constructs a noisy input:

$$x_t=\sqrt{\bar\alpha_t}\,x+\sqrt{1-\bar\alpha_t}\,\epsilon.$$

The U-Net predicts $\epsilon$ and is trained using noise-prediction MSE. During generation, we start from noise and let the scheduler use these predictions to take denoising steps. The clean image is unavailable. DDPM sampling adds randomness during the reverse process; DDIM with `eta=0` can sample deterministically from fixed initial noise.

**Flow matching here: learn a velocity.** We independently sample noise $x_0$ and a data point $x_1$, then construct a point on the path between them:

$$x_t=(1-t)x_0+t x_1,\qquad u=x_1-x_0.$$

Our network predicts $u$: the direction and rate of motion along this training path. During generation, it only receives the current point and time. A numerical solver integrates its predicted velocity; our midpoint solver evaluates the network twice per step. It does not know the data endpoint or subtract a known noise sample.

| | Lab 1 diffusion | This flow-matching model |
| --- | --- | --- |
| Training question | What noise was added? | What velocity moves this training pair along its path? |
| Target | $\epsilon$ | $x_1-x_0$ |
| Generation update | A scheduler converts noise predictions into denoising steps | A solver integrates predicted velocities |
| Time direction used here | High noise at large $t$; generate toward small $t$ | Noise at $t=0$; generate toward $t=1$ |

The straight lines belong to the **paired training examples**. Many pairs can pass through similar positions, so MSE trains the network toward their conditional average velocity. Its generated trajectories can therefore curve, as the plot above shows.

Either approach can operate on image latents. The VAE supplies the compressed representation and decodes the final generated latent; it does not determine whether the generative model uses diffusion or flow matching.
